1. Activate .venv environnement "source ./.venv/bin/activate"
2. Provide the path for the DeepSeek and GPT API keys
3. Insert a copy of Fowler's "Refactoring: Improve the design of existing code" 2nd edition inside the "Data" folder as "Fowler.pdf"

In [ ]:
from pandas.core.config_init import pc_width_doc

#TODO fill api keys paths
gpt_api_key_path = "src/generator/OpenAI_key.txt" #i.e. ""/../OpenAI_key.txt""
deepseek_api_key_path = "src/generator/DeepSeek_key.txt"

We start by extracting the content Fowler's book into a JSON

In [13]:
import sys
import os
sys.path.insert(0, os.path.abspath("../../.."))
print(os.getcwd())

/Users/jeancarlorspaul/Documents/Doc_Carl/Poly/Projects/Refactoring_LLM_Benchmark/src/scripts


In [16]:
import pathlib
for p in [
    "../../src/__init__.py",
    "../../src/generator/__init__.py",
    "../../src/generator/scripts/__init__.py"
]:
    pathlib.Path(p).touch(exist_ok=True)

In [17]:
from src.generator.scripts.pdf_extractor import fowlerpdf_to_json

fowlerpdf_to_json()

ModuleNotFoundError: No module named 'constants'

Next step is to prompt both GPT and DeepSeek and save their output inside JSON files. We run each prompt 5 times, but the value can be modified.

We also clean to ouput of the LLMs.

In [ ]:
from tqdm import tqdm
from prompt_generator import generate_llm_json, clean_llm_output

NB_RUNS = 5
GPT_RUN_OUTPUT_PREFIX = "gpt_run#"
DEEPSEEK_RUN_OUTPUT_PREFIX = "ds_run#"

GPT_MODEL_NAME = "gpt-4o-mini"
DEEPSEEK_MODEL_NAME = "deepseek-chat"

for x in tqdm(range(NB_RUNS)):
    gpt_filename = GPT_RUN_OUTPUT_PREFIX + str(x) + ".json"
    ds_filename = DEEPSEEK_RUN_OUTPUT_PREFIX + str(x) + ".json"

    generate_llm_json(filename=gpt_filename, api_key_path=gpt_api_key_path, model_name=GPT_MODEL_NAME)
    generate_llm_json(filename=ds_filename, api_key_path= deepseek_api_key_path, model_name=DEEPSEEK_MODEL_NAME)

    clean_llm_output(filename=gpt_filename)
    clean_llm_output(filename=ds_filename)


!!! NEED TO ADD JC's PART HERE !!!

Next part is data analysis. We start with the metrics calculations. And the we proceed with CodeBLEU calculations.

In [ ]:
# When using pyccmetrics, we need version 0.20.1 of tree sitter
!pip install tree-sitter==0.20.1

In [ ]:
from metrics_calculator import combine_runs_into_json

combine_runs_into_json(nb_runs=NB_RUNS)

In [ ]:
# codebleu library requires version 0.23.1 of tree sitter
!pip install tree-sitter==0.23.1

In [ ]:
from codebleu_calculator import add_codebleu_to_raw_data

add_codebleu_to_raw_data(nb_runs=NB_RUNS)